# Flow Matching on CIFAR10

Dieses Notebook zeigt einen einfachen End-to-End-Workflow fuer Continuous Flow Matching auf `CIFAR10`:

- Training eines FM-Modells
- Speichern und Laden des Checkpoints
- Erzeugen und Visualisieren generierter Samples

Die Implementierung nutzt die lokalen Helfer aus `flowguard.flow_matching` und dieselbe `CIFAR10`-Datengrundlage wie das restliche Framework.

### Referenz
Die Trainingsparameter in diesem Notebook orientieren sich am offiziellen
[facebookresearch/flow_matching CIFAR10-Beispiel](https://github.com/facebookresearch/flow_matching/tree/main/examples/image).

In [ ]:
import sys
from pathlib import Path

import torch
from IPython.display import Image as DisplayImage
from IPython.display import display


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the current notebook directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from flowguard.flow_matching.config import FlowMatchingTrainingConfig
from flowguard.flow_matching.training import (
    load_flow_matching_checkpoint,
    sample_images,
    train_flow_matching_model,
)

PROJECT_ROOT

## Training Config

Die Parameter unten entsprechen dem offiziellen CIFAR10-Training aus
`facebookresearch/flow_matching` (epochs=3000, batch_size=64, use_ema=True,
class_drop_prob=1.0 fuer unkonditioniertes Training).

Fuer einen schnellen Smoke-Test kann `epochs` auf 1 gesetzt werden.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "runs" / "flow_matching" / "cifar10_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config = FlowMatchingTrainingConfig(
    dataset="cifar10",
    data_path=str(PROJECT_ROOT / "data" / "cifar10"),
    output_dir=str(OUTPUT_DIR),
    batch_size=64,
    epochs=3000,
    lr=1e-4,
    num_workers=0,
    seed=0,
    device="cuda" if torch.cuda.is_available() else "cpu",
    class_drop_prob=1.0,
    use_ema=True,
    sample_every=100,
    sample_count=16,
    checkpoint_every=100,
    sample_ode_method="midpoint",
    sample_step_size=0.01,
    download=True,
)

config

## Train and Save

Diese Zelle trainiert das FM-Modell, speichert `checkpoint_latest.pt` im `output_dir` und schreibt automatisch Sample-Grids nach `samples/`.

In [ ]:
checkpoint = train_flow_matching_model(config)
checkpoint_path = checkpoint.checkpoint_path
sample_dir = Path(config.output_dir) / "samples"
training_sample_paths = sorted(sample_dir.glob("epoch_*.png"))

print(f"Checkpoint saved to: {checkpoint_path}")
print(f"Training samples directory: {sample_dir}")
training_sample_paths[-1] if training_sample_paths else None

## Reload Checkpoint and Generate Samples

Hier wird der gespeicherte Checkpoint wieder geladen und ein zusaetzliches Sample-Grid erzeugt.

In [ ]:
reloaded = load_flow_matching_checkpoint(checkpoint_path, device=config.device)
manual_sample_path = Path(config.output_dir) / "samples" / "manual_sample.png"

sample_images(
    reloaded.model,
    reloaded.dataset_config,
    output_path=manual_sample_path,
    device=config.device,
    sample_count=16,
    ode_method=config.sample_ode_method,
    step_size=config.sample_step_size,
    seed=config.seed + 123,
)

manual_sample_path

## Visualize Generated Samples

In [ ]:
latest_training_sample = training_sample_paths[-1] if training_sample_paths else None

if latest_training_sample is not None:
    print(f"Last training snapshot: {latest_training_sample}")
    display(DisplayImage(filename=str(latest_training_sample)))

print(f"Manual sample grid: {manual_sample_path}")
display(DisplayImage(filename=str(manual_sample_path)))